# Enhanced FLAN-T5 Grammar Error Correction — Final Experiment (v2)

This notebook is the **final experiment** for the paper. It trains a single model — FLAN-T5 — pushed as far as our compute budget allows, and evaluates it on the in-domain C4-200M test set **and** on the standard public GEC benchmarks.

**What is enhanced over `deepfinalproject.ipynb` (v1):**

| # | Enhancement | Why |
|---|---|---|
| 1 | **Cleaned dataset** — the four noise categories we documented in v1 (repeated adjacent words in targets, `!!`/`??` punctuation runs, HTML/markup artefacts, mojibake) are now *filtered out at sampling time*, plus whitespace normalisation, exact deduplication, a misalignment filter, and a non-English filter | v1 showed every metric was a lower bound because the "clean" side of C4-200M is noisy. Cleaner targets = a higher ceiling. |
| 2 | **Same data budget as v1, all of it clean** — 120,000 kept pairs from *each* of the 10 shards = **1.2M pairs**, every one passing the filters | Matched scale turns v1→v2 into a *controlled* data-quality comparison, and keeps each run single-session — leaving quota for the copy-gate ablation. (`PER_SHARD = 1_000_000` scales this back up to 10M for a future data-scaling run.) |
| 3 | **Copy gate ported to T5** — the pointer-generator gate that made Model 3 (LSTM+attention) a precise editor is re-implemented on top of T5's cross-attention | GEC output is ~75% verbatim copy; an explicit copy pathway is the right inductive bias. |
| 4 | **Full splits, no subsampling** — train on all ~1.15M pairs, validate on the full 25k val set, and evaluate on the **full 25k test set** (v1 evaluated on a 5k sample) | 5× the statistical power on every reported number. |
| 5 | **Training upgrades** — label smoothing 0.1 (same trick that helped Model 3), cosine LR schedule with warm-up, and automatic **checkpoint-resume across Kaggle sessions** | Resume is a safety net at 1.2M scale, and what makes the 10M scale-up possible later. |
| 6 | **Standard benchmarks** — BEA-2019 dev (W&I+LOCNESS) with the *official* ERRANT span-based F0.5 via `errant_compare`, plus JFLEG (4-reference GLEU) and optional CoNLL-2014 | Reviewers expect numbers on a standard benchmark, not only on synthetic in-domain data. |

**Compute plan (Kaggle T4×2):** the whole notebook completes in **one batch session** (*Save Version → Save & Run All*, ~6.5–8 h): sampling+splitting ≈ 30 min, training ≈ 3.5–4 h (~9k optimizer steps — the same optimization budget as v1), full-test evaluation ≈ 1.5–2 h, benchmarks ≈ 30–40 min. The copy-gate **ablation** (`USE_COPY_GATE = False`) is a second identical session; both runs fit comfortably in one week of Kaggle quota. The `MAX_TRAIN_HOURS` auto-pause and checkpoint-resume machinery remain as a safety net: if a session is ever interrupted, attach the previous version's output as an input and re-run — splits are restored and training resumes from the newest checkpoint. To scale up later, set `PER_SHARD = 1_000_000` (10M pairs, ~3 sessions) — the multi-session workflow handles it automatically.

**Before the full run:** set `QUICK_TEST = True` in the configuration cell and run the whole notebook once (~30–45 min) to smoke-test the pipeline end-to-end. Then set it back to `False` for the real run.

In [ ]:
# Install notebook dependencies
!pip install -q transformers accelerate sentencepiece errant pyarrow
!python -m spacy download en_core_web_sm -q

In [ ]:
# Imports, configuration, and shared helpers (FLAN-T5-only notebook)
import os, sys, csv, json, time, math, random, re, gc, hashlib, shutil, subprocess, zipfile
import urllib.request, tarfile
from pathlib import Path
from collections import Counter
from difflib import SequenceMatcher
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from nltk.translate.gleu_score import corpus_gleu
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, T5ForConditionalGeneration,
                          DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments)
from transformers.modeling_outputs import Seq2SeqLMOutput

os.environ["WANDB_DISABLED"] = "true"

SEED = 42
OUT = Path("/kaggle/working")
C4_DIR = Path("/kaggle/input/datasets/dariocioni/c4200m")
NOTEBOOK_VERSION = "t5max_2026_06_12"

# ----------------------------- experiment configuration -----------------------------
QUICK_TEST = False                   # True = tiny end-to-end smoke test (~30-45 min)
MODEL_NAME = "google/flan-t5-small"  # "google/flan-t5-base" if budget allows (~4-5x cost)
USE_COPY_GATE = True                 # pointer-generator gate ported from Model 3
GATE_INIT_BIAS = 2.0                 # sigmoid(2.0) ~ 0.88 -> start generation-dominant,
                                     # so step 0 behaves like the pretrained model

PER_SHARD = 120_000 if not QUICK_TEST else 20_000     # kept pairs per shard; 10 shards -> 1.2M
                                     # (matched to v1's budget; 1_000_000 -> 10M scale-up run)
VAL_N = 25_000 if not QUICK_TEST else 2_000
TEST_N = 25_000 if not QUICK_TEST else 2_000

MIN_W, MAX_W = 4, 50                 # same length window as v1
MAX_LEN_DIFF = 10                    # |len(src)-len(tgt)| above this -> misaligned pair
MIN_BAG_OVERLAP = 0.5                # token-multiset overlap below this -> misaligned pair
MAX_NONASCII_RATIO = 0.25            # mostly-non-English pairs are dropped

PREFIX = "Correct the grammar in this sentence: "
MAX_INPUT_LEN = 128
MAX_TARGET_LEN = 128
PER_DEVICE_BATCH = 16
GRAD_ACCUM = 4
EPOCHS = 1
LEARNING_RATE = 3e-4
LABEL_SMOOTHING = 0.1                # same trick that helped Model 3
SRC_DROPOUT_P = 0.05                 # source-side word dropout (regularisation), as in v1
EVAL_SAVE_STEPS = 2500 if not QUICK_TEST else 200
MAX_TRAIN_HOURS = 9.5                # pause training cleanly before Kaggle's 12h batch kill

GEN_BATCH = 64
GEN_KWARGS = dict(max_length=128, num_beams=5, length_penalty=0.7,
                  no_repeat_ngram_size=3, early_stopping=True, min_length=1)
REFINE_MAX_EDITS = 3                 # iterative-refinement acceptance threshold, as in v1

RUN_BEA19 = True
RUN_JFLEG = True
RUN_CONLL14 = True

MODEL_DIR_NAME = "t5max_model"
CKPT_DIR = OUT / "t5max_ckpts"
MANIFEST_PATH = OUT / "t5max_training_manifest.json"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------------- helpers ----------------------------------------
def save_json(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2), encoding="utf-8")

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def file_signature(path):
    path = Path(path)
    if not path.exists():
        return None
    stat = path.stat()
    # name+size only: mtime changes when splits are restored from a previous
    # session's output, but identical deterministic parquet => identical size
    return {"name": path.name, "size": int(stat.st_size)}

def data_signature():
    return {name: file_signature(OUT / f"{name}.parquet") for name in ["train", "val", "test"]}

def token_edit(a, b):
    sm = SequenceMatcher(None, str(a).split(), str(b).split())
    return sum(max(i2 - i1, j2 - j1) for tag, i1, i2, j1, j2 in sm.get_opcodes() if tag != "equal")

def require_c4_shards():
    files = sorted(C4_DIR.glob("C4_200M.tsv-*"))
    if files:
        return files
    input_root = Path("/kaggle/input")
    if input_root.exists():
        files = sorted(input_root.rglob("C4_200M.tsv-*"))
        if files:
            print(f"Auto-discovered C4 shards under: {files[0].parent}")
            return files
    raise FileNotFoundError(
        f"No C4_200M.tsv-* shards found under {C4_DIR} or /kaggle/input. "
        "Attach the C4_200M Kaggle dataset before running this notebook."
    )

def find_resume_checkpoint():
    """Newest local checkpoint, else checkpoints attached from a previous
    session's saved output (so training can continue across Kaggle sessions)."""
    def latest(cands):
        cands = [d for d in cands if (d / "trainer_state.json").exists()]
        return max(cands, key=lambda d: int(d.name.split("-")[-1]), default=None)
    if CKPT_DIR.exists():
        local = latest(list(CKPT_DIR.glob("checkpoint-*")))
        if local:
            return str(local)
    input_root = Path("/kaggle/input")
    if input_root.exists():
        cand = latest(list(input_root.glob("*/t5max_ckpts/checkpoint-*")) +
                      list(input_root.glob("*/*/t5max_ckpts/checkpoint-*")))
        if cand:
            CKPT_DIR.mkdir(exist_ok=True)
            # copy ALL checkpoints from that session (newest + best-so-far)
            for sib in sorted(cand.parent.glob("checkpoint-*")):
                dest = CKPT_DIR / sib.name
                if not dest.exists():
                    print(f"Copying previous-session checkpoint {sib} -> {dest}")
                    shutil.copytree(sib, dest)
            return str(CKPT_DIR / cand.name)
    return None

def splits_match_config(base):
    """True if the splits under `base` match the CURRENT config (guards against
    e.g. reusing tiny QUICK_TEST splits after flipping QUICK_TEST off)."""
    try:
        return (pq.read_metadata(base / "val.parquet").num_rows == VAL_N and
                pq.read_metadata(base / "test.parquet").num_rows == TEST_N)
    except Exception:
        return False

def try_restore_splits():
    """Reuse train/val/test parquet from an attached previous-session output
    instead of re-sampling. The pipeline is deterministic, so restored bytes
    are identical to a fresh re-run."""
    needed = ["train.parquet", "val.parquet", "test.parquet"]
    if all((OUT / n).exists() for n in needed):
        if splits_match_config(OUT):
            return True
        print("Existing splits do not match the current config (QUICK_TEST changed?) -- resampling.")
        return False
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for d in list(input_root.glob("*")) + list(input_root.glob("*/*")):
            if d.is_dir() and all((d / n).exists() for n in needed) and splits_match_config(d):
                print(f"Restoring data splits from {d}")
                for n in needed:
                    shutil.copy2(d / n, OUT / n)
                for extra in ["c4_gec_clean_meta.json", "t5max_eda_stats.json"]:
                    if (d / extra).exists() and not (OUT / extra).exists():
                        shutil.copy2(d / extra, OUT / extra)
                return True
    return False

# ---------------------------------------------------------------------------
# Edit-level F0.5 via ERRANT (lazy-loaded). Falls back to a word-level
# Levenshtein alignment if ERRANT or spaCy is unavailable. Compared to v1 the
# loop parses each source sentence ONCE (not twice) and skips parsing when the
# prediction is identical to the source -- about 2x faster on the full test set.
# ---------------------------------------------------------------------------
_ERRANT = {"loaded": False, "annotator": None, "error": None}

def _try_load_errant():
    if _ERRANT["loaded"]:
        return _ERRANT["annotator"]
    try:
        import errant
        _ERRANT["annotator"] = errant.load("en")
    except Exception as exc:
        _ERRANT["annotator"] = None
        _ERRANT["error"] = repr(exc)
    _ERRANT["loaded"] = True
    return _ERRANT["annotator"]

def _annotate_edits(annotator, src_doc, tgt_doc):
    out = []
    for e in annotator.annotate(src_doc, tgt_doc):
        if e.type == "noop":
            continue
        out.append((int(e.o_start), int(e.o_end), str(e.c_str).strip()))
    return out

def _word_align_edits(src, target):
    s = str(src).split()
    t = str(target).split()
    sm = SequenceMatcher(a=s, b=t, autojunk=False)
    edits = []
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            continue
        edits.append((i1, i2, " ".join(t[j1:j2])))
    return edits

def compute_fbeta_edit_metrics(src_strs, tgt_strs, pred_strs, beta=0.5, progress=True):
    annotator = _try_load_errant()
    use_errant = annotator is not None
    extractor = "errant" if use_errant else "levenshtein"
    tp = fp = fn = 0
    it = zip(src_strs, tgt_strs, pred_strs)
    if progress:
        it = tqdm(it, total=len(src_strs), desc=f"edit-F{beta:g} ({extractor})")
    for src, tgt, pred in it:
        src, tgt, pred = str(src), str(tgt), str(pred)
        if use_errant:
            try:
                src_doc = annotator.parse(src)
                gold = set() if tgt == src else set(_annotate_edits(annotator, src_doc, annotator.parse(tgt)))
                guess = set() if pred == src else set(_annotate_edits(annotator, src_doc, annotator.parse(pred)))
            except Exception:
                gold = set(_word_align_edits(src, tgt))
                guess = set(_word_align_edits(src, pred))
                extractor = "levenshtein_fallback"
        else:
            gold = set(_word_align_edits(src, tgt))
            guess = set(_word_align_edits(src, pred))
        tp += len(gold & guess)
        fp += len(guess - gold)
        fn += len(gold - guess)
    P = tp / max(tp + fp, 1)
    R = tp / max(tp + fn, 1)
    if (beta * beta * P + R) > 0:
        Fb = (1 + beta * beta) * P * R / (beta * beta * P + R)
    else:
        Fb = 0.0
    return {
        "precision_edit": float(P),
        "recall_edit": float(R),
        f"f{beta:g}_edit": float(Fb),
        "n_gold_edits": int(tp + fn),
        "n_pred_edits": int(tp + fp),
        "n_tp_edits": int(tp),
        "edit_extractor": extractor,
    }

def evaluate_predictions(model_name, src_strs, tgt_strs, pred_strs, out_json, out_csv,
                         extra=None, compute_f05=True):
    src_toks = [str(s).split() for s in src_strs]
    tgt_toks = [str(t).split() for t in tgt_strs]
    pred_toks = [str(p).split() for p in pred_strs]

    gleu_pred = corpus_gleu([[t] for t in tgt_toks], pred_toks)
    gleu_src = corpus_gleu([[t] for t in tgt_toks], src_toks)
    bleu_pred = corpus_bleu([[t] for t in tgt_toks], pred_toks, smoothing_function=SmoothingFunction().method1)
    exact = np.mean([str(p).strip() == str(t).strip() for p, t in zip(pred_strs, tgt_strs)])
    ed_src = np.mean([token_edit(s, t) for s, t in zip(src_strs, tgt_strs)])
    ed_pred = np.mean([token_edit(p, t) for p, t in zip(pred_strs, tgt_strs)])

    rows = []
    for s, t, p in zip(src_strs, tgt_strs, pred_strs):
        d_src = token_edit(s, t)
        d_pred = token_edit(p, t)
        if str(p).strip() == str(t).strip():
            category = "perfect"
        elif str(p).strip() == str(s).strip():
            category = "unchanged"
        elif d_pred < d_src:
            category = "improved"
        else:
            category = "harm"
        rows.append((s, t, p, category, d_src, d_pred))

    examples = pd.DataFrame(rows, columns=["src", "tgt", "pred", "category", "edit_src_to_tgt", "edit_pred_to_tgt"])
    examples.to_csv(OUT / out_csv, index=False)

    metrics = {
        "model": model_name,
        "notebook_version": NOTEBOOK_VERSION,
        "n_eval": int(len(src_strs)),
        "gleu_src_baseline": float(gleu_src),
        "gleu_pred": float(gleu_pred),
        "bleu_pred": float(bleu_pred),
        "exact_match": float(exact),
        "edit_src": float(ed_src),
        "edit_pred": float(ed_pred),
        "edit_reduction_pct": float((ed_src - ed_pred) / ed_src * 100) if ed_src else 0.0,
        "perfect_pct": float((examples["category"] == "perfect").mean() * 100),
        "improved_pct": float((examples["category"] == "improved").mean() * 100),
        "unchanged_pct": float((examples["category"] == "unchanged").mean() * 100),
        "harm_pct": float((examples["category"] == "harm").mean() * 100),
        "helpful_pct": float(((examples["category"] == "perfect") | (examples["category"] == "improved")).mean() * 100),
    }

    if compute_f05:
        metrics.update(compute_fbeta_edit_metrics(src_strs, tgt_strs, pred_strs, beta=0.5))

    if extra:
        metrics.update(extra)
    save_json(metrics, OUT / out_json)

    f05_disp = metrics.get("f0.5_edit")
    f05_str = f" | F0.5={f05_disp*100:.2f}%" if f05_disp is not None else ""
    print(f"{model_name} on {len(src_strs):,} examples")
    print(f"GLEU={gleu_pred:.4f} (src baseline {gleu_src:.4f}) | BLEU={bleu_pred:.4f} | "
          f"exact={exact*100:.2f}% | helpful={metrics['helpful_pct']:.1f}% | harm={metrics['harm_pct']:.1f}%{f05_str}")
    print("Behaviour (%):")
    print(examples["category"].value_counts(normalize=True).mul(100).round(1).to_string())
    print(f"Saved: {out_json}, {out_csv}")
    return metrics, examples

print(f"Config OK | QUICK_TEST={QUICK_TEST} | model={MODEL_NAME} | copy_gate={USE_COPY_GATE} | "
      f"target pairs={PER_SHARD * 10:,} | val={VAL_N:,} | test={TEST_N:,}")

## Data Preparation — 1.2M cleaned pairs

v1's *Final Challenge* analysis documented that C4-200M's "clean" target side contains systematic residual noise, in four categories:

- **A — repeated adjacent words** surviving in the target (*"the NEW new mini"*),
- **B — excessive punctuation** (*"Love your shots!! Those are amazing!"*),
- **C — abnormal spacing** (multiple internal spaces),
- **D — markup / encoding artefacts** (HTML fragments, mojibake, U+FFFD).

Since the target is the supervision signal, noisy targets directly teach the model to *produce* noise, and they corrupt every edit-level metric. Here we attack this at sampling time:

1. **Normalise** (not drop): whitespace runs are collapsed to single spaces on both sides — fixes category C without losing data.
2. **Drop** any pair whose target shows category A, B, or D noise, or whose *source* shows markup/mojibake.
3. **Drop** mostly-non-ASCII pairs (multilingual leakage into an English GEC corpus).
4. **Drop misaligned pairs**: if the source and target differ in length by more than 10 words, or share less than 50% of their tokens (multiset overlap), the pair is almost certainly a corpus alignment error, not a grammar correction (v1 example: the mangled *"International Tax. Honduras"* pair).
5. **Deduplicate** on exact source match across all shards (C4 contains many crawl duplicates).
6. Keep v1's filters: 4–50 words on both sides, `src != tgt`.

We keep scanning each shard until **120,000 pairs survive all filters**, giving 1.2M pairs total — the *same data budget as v1*, but with every pair passing the noise filters. Matching v1's scale keeps each run single-session and makes the v1→v2 comparison controlled: same quantity, different quality. (Setting `PER_SHARD = 1_000_000` scales this to 10M for a future data-scaling experiment.) The trade-off is acknowledged in the paper: filtering repeated adjacent words also removes the rare legitimate construction (*"had had"*, *"that that"*), a small price for removing a systematic error source.

In [ ]:
# Noise filters (target the four noise categories documented in v1)
_RE_WS = re.compile(r"\s+")
_RE_TAG = re.compile(r"<[a-zA-Z/!][^>]*>")                      # HTML/XML-ish tags
_RE_ENTITY = re.compile(r"&[a-zA-Z]{2,6};|&#\d+;")              # &amp; &lt; &#39; ...
_RE_URL = re.compile(r"https?://|www\.|href=|mailto:", re.IGNORECASE)
_RE_PUNCT_RUN = re.compile(r"[!?]{2,}|\.{4,}|,{2,}|&&|;;")      # !! ?? ,, && ;; .....
_RE_MOJIBAKE = re.compile("�|â€|ï¿½")  # U+FFFD and broken UTF-8

def normalize_ws(s):
    return _RE_WS.sub(" ", s).strip()

def has_repeated_adjacent_word(s):
    toks = s.lower().split()
    return any(a == b and any(ch.isalnum() for ch in a) for a, b in zip(toks, toks[1:]))

def nonascii_ratio(s):
    if not s:
        return 0.0
    return sum(ord(c) > 127 for c in s) / len(s)

def bag_overlap(src_toks, tgt_toks):
    cs, ct = Counter(src_toks), Counter(tgt_toks)
    inter = sum((cs & ct).values())
    return inter / max(len(src_toks), len(tgt_toks), 1)

def pair_noise_reason(src, tgt):
    """Return None if the pair is clean, else a short drop-reason string."""
    for s, side in ((src, "src"), (tgt, "tgt")):
        if _RE_TAG.search(s) or _RE_ENTITY.search(s) or _RE_URL.search(s):
            return f"markup_{side}"
        if _RE_MOJIBAKE.search(s):
            return f"mojibake_{side}"
    if _RE_PUNCT_RUN.search(tgt):
        return "punct_run_tgt"
    if has_repeated_adjacent_word(tgt):
        return "repeat_word_tgt"
    if nonascii_ratio(tgt) > MAX_NONASCII_RATIO:
        return "nonascii_tgt"
    st, tt = src.split(), tgt.split()
    if abs(len(st) - len(tt)) > MAX_LEN_DIFF:
        return "len_diff"
    if bag_overlap(st, tt) < MIN_BAG_OVERLAP:
        return "low_overlap"
    return None

# Sanity check on real noise examples found in v1 (dataset_noise_examples.csv)
demo = [
    ("What do you think of the NEW new minis?",
     "What do you think of the NEW new mini?"),                       # category A
    ("I really lile this!! You are super talented!!",
     "I really like this!! You are super talented!!"),                # category B
    ("Title                   : Why You it Should be Snakes?",
     "Title                   : Why Did it Have to be Snakes?"),      # category C (normalised, kept)
    ("Contact <TT><A HREF=\"mailto:web@perl.com\">web@perl.com</A></TT> for details.",
     "Contact web@perl.com for details please now."),                 # category D
    ("She go to the market yesterday morning.",
     "She went to the market yesterday morning."),                    # clean pair
]
print(f"{'verdict':28s}  target")
for src, tgt in demo:
    src_n, tgt_n = normalize_ws(src), normalize_ws(tgt)
    reason = pair_noise_reason(src_n, tgt_n)
    verdict = f"DROP ({reason})" if reason else "KEEP"
    print(f"{verdict:28s}  {tgt_n[:70]}")

In [ ]:
# Sample PER_SHARD clean pairs from EVERY C4_200M shard (10 x 120k = 1.2M total,
# the same data budget as v1 but fully cleaned).
# If train/val/test splits already exist (or are restored from an attached
# previous-session output), sampling is skipped entirely (~40-50 min saved).
if try_restore_splits():
    print("train/val/test splits available -- skipping sampling.")
else:
    files = require_c4_shards()
    print(f"Shards: {len(files)} | target kept per shard: {PER_SHARD:,}")

    stats = Counter()
    seen_src = set()          # exact-duplicate sources across all shards
    shard_paths = []
    t0 = time.time()

    for i, fpath in enumerate(files):
        kept = 0
        src_buf, tgt_buf = [], []
        with open(fpath, "r", encoding="utf-8", errors="replace") as fh:
            for line in fh:
                stats["scanned"] += 1
                cols = line.rstrip("\r\n").split("\t")
                if len(cols) != 2:
                    stats["drop_format"] += 1
                    continue
                src, tgt = normalize_ws(cols[0]), normalize_ws(cols[1])
                if not src or not tgt or src == tgt:
                    stats["drop_identical"] += 1
                    continue
                ls, lt = len(src.split()), len(tgt.split())
                if not (MIN_W <= ls <= MAX_W and MIN_W <= lt <= MAX_W):
                    stats["drop_length"] += 1
                    continue
                h = hash(src)
                if h in seen_src:
                    stats["drop_duplicate"] += 1
                    continue
                reason = pair_noise_reason(src, tgt)
                if reason is not None:
                    stats["drop_" + reason] += 1
                    continue
                seen_src.add(h)
                src_buf.append(src)
                tgt_buf.append(tgt)
                kept += 1
                stats["kept"] += 1
                if kept >= PER_SHARD:
                    break
        if kept < PER_SHARD:
            print(f"WARNING: shard {i} exhausted at {kept:,} pairs (< {PER_SHARD:,})")
        shard_path = OUT / f"c4_clean_shard_{i:02d}.parquet"
        tbl = pa.table({"src": pa.array(src_buf, type=pa.large_string()),
                        "tgt": pa.array(tgt_buf, type=pa.large_string())})
        pq.write_table(tbl, shard_path)
        shard_paths.append(shard_path)
        del src_buf, tgt_buf, tbl
        gc.collect()
        print(f"Shard {i}: kept {kept:,} | total {stats['kept']:,} | elapsed {time.time()-t0:.0f}s")

    del seen_src
    gc.collect()

    print("\nFilter stats:")
    for k in sorted(stats, key=stats.get, reverse=True):
        print(f"  {k:24s} {stats[k]:,}")
    keep_rate = stats["kept"] / max(stats["scanned"], 1) * 100
    print(f"\nKeep rate: {keep_rate:.1f}% | total kept: {stats['kept']:,}")

    meta = {
        "notebook_version": NOTEBOOK_VERSION,
        "n_pairs": int(stats["kept"]),
        "per_shard": PER_SHARD,
        "min_words": MIN_W, "max_words": MAX_W,
        "max_len_diff": MAX_LEN_DIFF,
        "min_bag_overlap": MIN_BAG_OVERLAP,
        "max_nonascii_ratio": MAX_NONASCII_RATIO,
        "shards_used": [f.name for f in files],
        "filter_stats": dict(stats),
    }
    save_json(meta, OUT / "c4_gec_clean_meta.json")
    print("Saved: c4_gec_clean_meta.json")

In [ ]:
# Train/val/test split (full-size splits) + EDA
_splits_ok = all((OUT / f"{n}.parquet").exists() for n in ("train", "val", "test"))
if _splits_ok and not splits_match_config(OUT):
    _splits_ok = False
if _splits_ok:
    for n in ("train", "val", "test"):
        rows = pq.read_metadata(OUT / f"{n}.parquet").num_rows
        print(f"{n}: {rows:,} pairs (existing)")
    print("Splits already exist -- skipping split + EDA "
          "(EDA artifacts were produced in the session that created them).")
else:
    shard_paths = sorted(OUT.glob("c4_clean_shard_*.parquet"))
    full = pa.concat_tables([pq.read_table(p) for p in shard_paths])
    n = full.num_rows
    print(f"Total clean pairs: {n:,}")

    rng = np.random.default_rng(SEED)
    perm = rng.permutation(n)
    test_idx = perm[:TEST_N]
    val_idx = perm[TEST_N:TEST_N + VAL_N]
    train_idx = perm[TEST_N + VAL_N:]

    for name, idx in (("train", train_idx), ("val", val_idx), ("test", test_idx)):
        tbl = full.take(pa.array(idx))
        pq.write_table(tbl, OUT / f"{name}.parquet")
        print(f"{name}: {tbl.num_rows:,} pairs -> {name}.parquet")
        del tbl
        gc.collect()
    del full
    gc.collect()

    # EDA on a sample of the train split (lengths on 1M, edit distance on 50k)
    train_tbl = pq.read_table(OUT / "train.parquet")
    eda_rng = np.random.default_rng(SEED)
    samp_n = min(1_000_000, train_tbl.num_rows)
    sub = train_tbl.take(pa.array(eda_rng.choice(train_tbl.num_rows, samp_n, replace=False))).to_pandas()
    del train_tbl
    gc.collect()

    src_lens = sub["src"].str.split().str.len().to_numpy()
    tgt_lens = sub["tgt"].str.split().str.len().to_numpy()
    diff = tgt_lens - src_lens

    ed_n = min(50_000, len(sub))
    ed_sub = sub.iloc[:ed_n]
    edits = np.array([token_edit(s, t) for s, t in zip(ed_sub["src"], ed_sub["tgt"])])

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].hist(src_lens, bins=50, color="#cc4444", alpha=0.75)
    axes[0].set_title("Source (corrupted) length"); axes[0].set_xlabel("words"); axes[0].set_ylabel("count"); axes[0].grid(alpha=0.3)
    axes[1].hist(tgt_lens, bins=50, color="#4477cc", alpha=0.75)
    axes[1].set_title("Target (clean) length"); axes[1].set_xlabel("words"); axes[1].grid(alpha=0.3)
    axes[2].hist(diff, bins=range(-12, 13), color="#666666", alpha=0.75)
    axes[2].set_title("Length difference (tgt - src)"); axes[2].set_xlabel("words"); axes[2].grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(OUT / "t5max_01_length_distributions.png", dpi=120, bbox_inches="tight"); plt.show()

    plt.figure(figsize=(8, 4))
    plt.hist(edits, bins=range(0, 21), color="#cc8844", alpha=0.8, edgecolor="black", linewidth=0.3)
    plt.xlabel("token edit distance"); plt.ylabel("count")
    plt.title(f"Token edit distance ({ed_n // 1000}k sample)  mean={edits.mean():.2f}  median={int(np.median(edits))}  p95={int(np.percentile(edits, 95))}")
    plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(OUT / "t5max_02_edit_distance.png", dpi=120, bbox_inches="tight"); plt.show()

    print(f"src words : mean={src_lens.mean():.2f}  median={np.median(src_lens):.0f}  p95={np.percentile(src_lens, 95):.0f}")
    print(f"tgt words : mean={tgt_lens.mean():.2f}  median={np.median(tgt_lens):.0f}  p95={np.percentile(tgt_lens, 95):.0f}")
    print(f"edits     : mean={edits.mean():.2f}  median={int(np.median(edits))}  p95={int(np.percentile(edits, 95))}")

    eda = {"sample_n": int(samp_n),
           "src_len_mean": float(src_lens.mean()), "src_len_p95": int(np.percentile(src_lens, 95)),
           "tgt_len_mean": float(tgt_lens.mean()), "tgt_len_p95": int(np.percentile(tgt_lens, 95)),
           "edit_mean": float(edits.mean()), "edit_median": int(np.median(edits)),
           "edit_p95": int(np.percentile(edits, 95))}
    save_json(eda, OUT / "t5max_eda_stats.json")
    del sub
    gc.collect()
    print("Saved: train/val/test parquet, t5max_01_length_distributions.png, t5max_02_edit_distance.png, t5max_eda_stats.json")

## Porting Model 3's copy gate to T5

The pointer-generator gate was the reason Model 3 (LSTM + Bahdanau attention) became a *precise* editor (edit precision 0.448 at only 13% harm). The same inductive bias should help T5: in GEC, ~75% of output tokens are verbatim copies of source tokens, so the model benefits from an explicit copy pathway instead of having to regenerate every token from its 32k-token vocabulary.

**Mechanism.** At every decoder position the output distribution becomes a learned mixture:

$$P(w) \;=\; p_{gen}\cdot \mathrm{softmax}(\text{LM head})_w \;+\; (1-p_{gen})\sum_{i:\,src_i = w} \alpha_i$$

where $\alpha$ is the head-averaged **cross-attention of the last decoder layer** (T5's counterpart of Bahdanau attention weights), and the gate is

$$p_{gen} = \sigma\!\big(W_g\,[\,h^{dec}_t \,;\, c_t\,] + b_g\big), \qquad c_t = \textstyle\sum_i \alpha_i\, h^{enc}_i .$$

The copy distribution is built exactly as in Model 3: each source position's attention weight is scatter-added onto the vocabulary id of the token sitting at that position.

**Three implementation details that make this work cleanly inside Hugging Face:**

1. **Initialisation:** $W_g = 0$, $b_g = +2$, so $p_{gen} \approx 0.88$ at step 0 — the model starts as (almost exactly) the pre-trained FLAN-T5 and *learns* when copying helps, instead of having an untrained copy path destroy the pre-trained behaviour.
2. **Log-probabilities as logits:** the forward pass returns $\log P(w)$ in place of logits. Because the mixture already sums to (approximately) 1, `log_softmax` over it is the identity — so beam search, `no_repeat_ngram`, and the Trainer's label smoothing all work unchanged, and the cross-entropy computed on these "logits" is exactly the NLL of the mixture.
3. **Beam search plumbing:** the encoder's `input_ids` are passed to `generate()` as an extra tensor (`copy_src_ids`); HF automatically expands it across beams together with `encoder_outputs`.

The mixture is computed in float32 for numerical stability under fp16 training. The instruction prefix tokens are technically copyable too, which is harmless: the gate learns to ignore them. Setting `USE_COPY_GATE = False` in the config cell gives the vanilla fine-tuned T5 — run both settings for the **ablation table** in the paper.

In [ ]:
%%writefile /kaggle/working/t5_copy_gate.py
# T5 with a pointer-generator copy gate (port of Model 3's mechanism).
# This class lives in a real module file (not the notebook) because recent
# transformers versions inspect the defining module's source file during
# from_pretrained; classes defined in a notebook (__main__) have no __file__.
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import T5ForConditionalGeneration
from transformers.modeling_outputs import Seq2SeqLMOutput


class T5WithCopyGate(T5ForConditionalGeneration):
    """FLAN-T5 + pointer-generator copy gate.

    P(w) = p_gen * softmax(lm_logits)[w] + (1 - p_gen) * sum_{i: src_i == w} attn_i
    attn  = head-averaged cross-attention of the last decoder layer
    p_gen = sigmoid(W_g [decoder_state ; attention_context] + b_g)

    The returned `logits` are log-probabilities of the mixture. The mixture sums
    to ~1, so log_softmax over it is (numerically) the identity: HF generate(),
    beam search, and the Trainer's label smoothing work unchanged.
    """

    def __init__(self, config):
        super().__init__(config)
        self.copy_gate = nn.Linear(2 * config.d_model, 1)
        nn.init.zeros_(self.copy_gate.weight)
        nn.init.constant_(self.copy_gate.bias, 2.0)
        self._last_pgen_mean = None

    def forward(self, input_ids=None, attention_mask=None, decoder_input_ids=None,
                labels=None, copy_src_ids=None, **kwargs):
        if labels is not None and decoder_input_ids is None and kwargs.get("decoder_inputs_embeds") is None:
            decoder_input_ids = self._shift_right(labels)
        kwargs["output_attentions"] = True
        kwargs["output_hidden_states"] = True
        kwargs["return_dict"] = True
        outputs = super().forward(input_ids=input_ids, attention_mask=attention_mask,
                                  decoder_input_ids=decoder_input_ids, labels=None, **kwargs)

        if copy_src_ids is None:
            copy_src_ids = input_ids

        if copy_src_ids is None:
            # No source ids available (plain LM usage): keep the vanilla distribution.
            new_logits = outputs.logits
        else:
            attn = outputs.cross_attentions[-1].mean(dim=1).float()        # (B, T, S)
            if attention_mask is not None:
                attn = attn * attention_mask.unsqueeze(1).float()          # zero padded src
            enc_h = outputs.encoder_last_hidden_state.float()              # (B, S, D)
            dec_h = outputs.decoder_hidden_states[-1].float()              # (B, T, D)
            context = torch.bmm(attn, enc_h)                               # (B, T, D)
            p_gen = torch.sigmoid(self.copy_gate(torch.cat([dec_h, context], dim=-1)).float())
            self._last_pgen_mean = float(p_gen.mean().detach())

            probs = torch.softmax(outputs.logits.float(), dim=-1)          # (B, T, V)
            probs = probs * p_gen
            copy_w = attn * (1.0 - p_gen)                                  # (B, T, S)
            idx = copy_src_ids.unsqueeze(1).expand(-1, probs.size(1), -1)  # (B, T, S)
            probs.scatter_add_(2, idx, copy_w)
            new_logits = torch.log(probs + 1e-9)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(new_logits.view(-1, new_logits.size(-1)),
                                   labels.view(-1).to(new_logits.device), ignore_index=-100)

        return Seq2SeqLMOutput(
            loss=loss,
            logits=new_logits,
            past_key_values=outputs.past_key_values,
            decoder_hidden_states=outputs.decoder_hidden_states,
            decoder_attentions=outputs.decoder_attentions,
            cross_attentions=outputs.cross_attentions,
            encoder_last_hidden_state=outputs.encoder_last_hidden_state,
            encoder_hidden_states=outputs.encoder_hidden_states,
            encoder_attentions=outputs.encoder_attentions,
        )

    def prepare_inputs_for_generation(self, input_ids, **kwargs):
        model_inputs = super().prepare_inputs_for_generation(input_ids, **kwargs)
        if kwargs.get("copy_src_ids") is not None:
            model_inputs["copy_src_ids"] = kwargs["copy_src_ids"]
        return model_inputs

In [ ]:
# Import the copy-gate class from the module file written by the cell above
import importlib
if str(OUT) not in sys.path:
    sys.path.insert(0, str(OUT))
import t5_copy_gate
t5_copy_gate = importlib.reload(t5_copy_gate)  # pick up changes on notebook re-runs
from t5_copy_gate import T5WithCopyGate
print(f"T5WithCopyGate imported from {t5_copy_gate.__file__}")

## Training — full cleaned train set, single session

Setup mirrors v1 where it worked (AdamW, lr 3e-4, effective batch 128 on T4×2, fp16, 6% warm-up, source-word dropout 0.05) and upgrades what v1 lacked:

- **Label smoothing 0.1** — the same regularisation that helped Model 3; also the T5 paper's default.
- **Cosine LR schedule** — smoother decay than linear over a single epoch.
- **Whole training set** — every clean pair is seen exactly once (~9k optimizer steps at effective batch 128, the *same optimization budget as v1's run* — so metric differences are attributable to the data cleaning and the copy gate, not to longer training).
- **Session-budget safety net** — a `TimeBudgetCallback` stops training cleanly after `MAX_TRAIN_HOURS` (default 9.5 h) and saves a checkpoint; downstream cells detect the pause and skip gracefully. At 1.2M scale this never triggers, but it is what makes the `PER_SHARD = 1_000_000` (10M-pair, ~3-session) scale-up safe: a Kaggle batch run killed at the hard 12 h limit saves **no** output.
- **Resume** — on restart, the newest checkpoint in `/kaggle/working/t5max_ckpts/` or in any attached previous-session output is picked up automatically; the data pipeline is deterministic (seeded), so a resumed run sees the identical sample order.

**RAM note:** the dataset object keeps the pairs in a memory-efficient pyarrow table and tokenizes on the fly — no large pandas object columns; this is also what makes the 10M scale-up fit in Kaggle RAM.

**Precision note:** T5 is known to occasionally produce NaNs under fp16; v1's fp16 run on T4 completed without issue, so we keep fp16 (bf16 is used automatically when the GPU supports it). If you ever see NaN loss, set `fp16=False` and accept the ~2× slowdown.

In [ ]:
# Fine-tune FLAN-T5 (with optional copy gate) on the FULL cleaned train set.
# A time-budget callback pauses training cleanly before Kaggle's 12h batch kill;
# the next session resumes from the newest checkpoint automatically.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from transformers import TrainerCallback

class TimeBudgetCallback(TrainerCallback):
    """Stop training (with a checkpoint save) after MAX_TRAIN_HOURS so the
    session never hits Kaggle's hard 12h kill. Resumed next session."""
    def __init__(self, max_hours):
        self.max_seconds = max_hours * 3600
        self.t0 = None
        self.stopped_early = False

    def on_train_begin(self, args, state, control, **kwargs):
        self.t0 = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        if time.time() - self.t0 > self.max_seconds:
            if not self.stopped_early:
                print(f"\nTime budget ({self.max_seconds / 3600:.1f}h) reached at step "
                      f"{state.global_step:,}/{state.max_steps:,} -- saving checkpoint and pausing.")
            self.stopped_early = True
            control.should_save = True
            control.should_training_stop = True
        return control

tokenizer_t5 = AutoTokenizer.from_pretrained(MODEL_NAME)
if USE_COPY_GATE:
    model_t5 = T5WithCopyGate.from_pretrained(MODEL_NAME)
    # re-apply the gate init explicitly (from_pretrained may have re-initialised it)
    model_t5.copy_gate.weight.data.zero_()
    model_t5.copy_gate.bias.data.fill_(GATE_INIT_BIAS)
else:
    model_t5 = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def _maybe_drop_words(text, p, rng):
    if p <= 0.0:
        return text
    words = str(text).split()
    if len(words) < 4:
        return text
    kept = [w for w in words if rng.random() > p]
    return " ".join(kept) if kept else text

class GECDataset(Dataset):
    """Pyarrow-backed: low RAM even at 10M rows; tokenizes on the fly."""
    def __init__(self, parquet_path, tokenizer, src_dropout_p=0.0, seed=SEED):
        tbl = pq.read_table(parquet_path)
        self.src = tbl.column("src")
        self.tgt = tbl.column("tgt")
        self.tokenizer = tokenizer
        self.src_dropout_p = src_dropout_p
        self.seed = seed

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        src = str(self.src[idx].as_py())
        tgt = str(self.tgt[idx].as_py())
        if self.src_dropout_p > 0:
            rng = random.Random(self.seed * 1_000_003 + idx)
            src = _maybe_drop_words(src, self.src_dropout_p, rng)
        enc = self.tokenizer(PREFIX + src, max_length=MAX_INPUT_LEN, truncation=True)
        labels = self.tokenizer(text_target=tgt, max_length=MAX_TARGET_LEN, truncation=True)
        enc["labels"] = labels["input_ids"]
        return enc

train_ds = GECDataset(OUT / "train.parquet", tokenizer_t5, src_dropout_p=SRC_DROPOUT_P, seed=SEED)
val_ds = GECDataset(OUT / "val.parquet", tokenizer_t5, src_dropout_p=0.0, seed=SEED + 1)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer_t5, model=model_t5, label_pad_token_id=-100)

n_gpus = max(1, torch.cuda.device_count())
eff_batch = PER_DEVICE_BATCH * n_gpus * GRAD_ACCUM
steps_per_epoch = math.ceil(len(train_ds) / eff_batch)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = max(1, int(total_steps * 0.06))
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

print(f"Fine-tuning {MODEL_NAME} | copy_gate={USE_COPY_GATE}")
print(f"Train pairs: {len(train_ds):,} (FULL train set) | Val pairs: {len(val_ds):,} (FULL val set)")
print(f"GPUs: {n_gpus} | effective batch: {eff_batch} | optimizer steps: {total_steps:,} | warmup: {warmup_steps:,}")
print(f"Session training budget: {MAX_TRAIN_HOURS}h")
print(f"Precision: {'bf16' if use_bf16 else ('fp16' if torch.cuda.is_available() else 'fp32')}")

manifest = {
    "notebook_version": NOTEBOOK_VERSION,
    "model_name": MODEL_NAME,
    "model_dir": MODEL_DIR_NAME,
    "use_copy_gate": USE_COPY_GATE,
    "gate_init_bias": GATE_INIT_BIAS if USE_COPY_GATE else None,
    "completed": False,
    "started_utc": datetime.now(timezone.utc).isoformat(),
    "data_signature": data_signature(),
    "train_rows": int(len(train_ds)),
    "val_rows": int(len(val_ds)),
    "src_dropout_p": SRC_DROPOUT_P,
    "label_smoothing": LABEL_SMOOTHING,
    "lr_scheduler": "cosine",
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS,
    "prefix": PREFIX,
    "quick_test": QUICK_TEST,
}
save_json(manifest, MANIFEST_PATH)

args_common = dict(
    output_dir=str(CKPT_DIR),
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    label_smoothing_factor=LABEL_SMOOTHING,
    logging_steps=500 if not QUICK_TEST else 50,
    save_steps=EVAL_SAVE_STEPS,
    eval_steps=EVAL_SAVE_STEPS,
    save_total_limit=2,
    predict_with_generate=False,
    fp16=torch.cuda.is_available() and not use_bf16,
    bf16=use_bf16,
    dataloader_num_workers=2,
    report_to=[],
    seed=SEED,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)
try:
    args = Seq2SeqTrainingArguments(evaluation_strategy="steps", save_strategy="steps", **args_common)
except TypeError:
    args = Seq2SeqTrainingArguments(eval_strategy="steps", save_strategy="steps", **args_common)

time_cb = TimeBudgetCallback(MAX_TRAIN_HOURS)
trainer = Seq2SeqTrainer(
    model=model_t5,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    callbacks=[time_cb],
)

resume = find_resume_checkpoint()
if resume:
    print(f"Resuming from checkpoint: {resume}")

t0 = time.time()
trainer.train(resume_from_checkpoint=resume)
print(f"T5 fine-tuning time this session: {(time.time() - t0) / 60:.1f} min")

log_history = trainer.state.log_history
save_json(log_history, OUT / "t5max_trainer_log_history.json")
train_logs = [(x.get("step"), x.get("loss")) for x in log_history if "loss" in x and x.get("step") is not None]
eval_logs = [(x.get("step"), x.get("eval_loss")) for x in log_history if "eval_loss" in x and x.get("step") is not None]
if train_logs or eval_logs:
    plt.figure(figsize=(9, 4))
    if train_logs:
        xs, ys = zip(*train_logs)
        plt.plot(xs, ys, label="train loss")
    if eval_logs:
        xs, ys = zip(*eval_logs)
        plt.plot(xs, ys, marker="o", label="eval loss")
    plt.title(f"{MODEL_NAME} (copy_gate={USE_COPY_GATE}) training on {len(train_ds):,} pairs")
    plt.xlabel("optimizer step"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT / "t5max_07_training_curves.png", dpi=120, bbox_inches="tight")
    plt.show()

finished = not time_cb.stopped_early
if not finished:
    manifest.update({
        "completed": False,
        "paused_at_step": int(trainer.state.global_step),
        "total_steps": int(trainer.state.max_steps),
        "paused_utc": datetime.now(timezone.utc).isoformat(),
    })
    save_json(manifest, MANIFEST_PATH)
    print("=" * 70)
    print(f"PAUSED at step {trainer.state.global_step:,}/{trainer.state.max_steps:,}.")
    print("Next steps:")
    print("  1. Let this batch run finish saving (it will, automatically).")
    print("  2. New session: Add Input -> Your Work -> this notebook's latest")
    print("     version output, then Save & Run All again. Splits are restored")
    print("     and training resumes from the newest checkpoint automatically.")
    print("=" * 70)
else:
    save_dir = OUT / MODEL_DIR_NAME
    trainer.save_model(str(save_dir))
    tokenizer_t5.save_pretrained(str(save_dir))

    # Diagnostic: where did the gate settle? (1.0 = pure generation, 0.0 = pure copy)
    if USE_COPY_GATE:
        model_t5.eval()
        probe = collator([val_ds[i] for i in range(min(32, len(val_ds)))])
        probe = {k: v.to(model_t5.device) for k, v in probe.items()}
        with torch.no_grad():
            model_t5(**probe)
        print(f"Mean p_gen on a validation batch: {model_t5._last_pgen_mean:.3f}")
        manifest["mean_p_gen_val_probe"] = model_t5._last_pgen_mean

    manifest.update({
        "completed": True,
        "finished_utc": datetime.now(timezone.utc).isoformat(),
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_metric": float(trainer.state.best_metric) if trainer.state.best_metric is not None else None,
    })
    save_json(manifest, MANIFEST_PATH)
    print(f"TRAINING COMPLETE. Saved fine-tuned model to: {save_dir}")

## In-domain evaluation — the FULL test set

Same decoding protocol as v1 (beam 5, length penalty 0.7, `no_repeat_ngram_size` 3, plus the guarded second refinement pass that only accepts ≤ 3-token changes), but evaluated on **all 25,000 test pairs** instead of a 5,000 sample. Metrics are identical to the paper's suite — GLEU, BLEU, exact match, behaviour categories (perfect / improved / unchanged / harm), and ERRANT edit-level P / R / F0.5 — so the numbers drop straight into the existing tables.

Note when comparing to v1: the v2 test set is drawn from the *cleaned* distribution, so the do-nothing source baseline will differ slightly from v1's 0.614. Always report the baseline alongside.

In [ ]:
# Evaluate on the FULL in-domain test set (beam search + guarded iterative refinement).
# Skips gracefully while training is paused mid-way (multi-session runs).
manifest = load_json(MANIFEST_PATH) if MANIFEST_PATH.exists() else None
if manifest is None:
    print("No training manifest found -- run the training cell first. Skipping evaluation.")
elif not manifest.get("completed"):
    print(f"Training paused at step {manifest.get('paused_at_step', '?')} of "
          f"{manifest.get('total_steps', '?')} -- not finished yet. Skipping evaluation.")
    print("Save Version now; next session attach this version's output and re-run all cells.")
elif manifest.get("data_signature") != data_signature():
    print("WARNING: train/val/test files changed after training -- skipping evaluation. Re-run training first.")
else:
    MODEL_DIR = OUT / manifest["model_dir"]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer_t5 = AutoTokenizer.from_pretrained(str(MODEL_DIR))
    model_cls = T5WithCopyGate if manifest.get("use_copy_gate") else AutoModelForSeq2SeqLM
    model_t5 = model_cls.from_pretrained(str(MODEL_DIR))
    model_t5.to(device)
    model_t5.eval()
    print(f"Loaded {MODEL_DIR} ({model_cls.__name__}) on {device}")

    def t5_correct_batch(texts, batch_size=GEN_BATCH, desc=None):
        out = []
        rng = range(0, len(texts), batch_size)
        if desc:
            rng = tqdm(rng, desc=desc)
        for i in rng:
            chunk = [PREFIX + str(s) for s in texts[i:i + batch_size]]
            enc = tokenizer_t5(chunk, return_tensors="pt", padding=True,
                               truncation=True, max_length=MAX_INPUT_LEN).to(device)
            kw = dict(GEN_KWARGS)
            if manifest.get("use_copy_gate"):
                kw["copy_src_ids"] = enc["input_ids"]
            with torch.no_grad():
                output_ids = model_t5.generate(**enc, **kw)
            out.extend(tokenizer_t5.batch_decode(output_ids, skip_special_tokens=True))
        return out

    test_tbl = pq.read_table(OUT / "test.parquet")
    src_strs = [str(x) for x in test_tbl.column("src").to_pylist()]
    tgt_strs = [str(x) for x in test_tbl.column("tgt").to_pylist()]
    print(f"Evaluating on the FULL test set: {len(src_strs):,} pairs")

    t0 = time.time()
    pass1 = t5_correct_batch(src_strs, desc="T5 pass 1")
    print(f"Pass 1 generated in {(time.time() - t0) / 60:.1f} min")

    # Iterative refinement: re-run the model on its own pass-1 outputs; accept only
    # refinements that change <= REFINE_MAX_EDITS tokens (stops cascading hallucinations).
    t1 = time.time()
    need = [i for i, p in enumerate(pass1) if p.strip() != src_strs[i].strip()]
    pred_strs = list(pass1)
    if need:
        refined = t5_correct_batch([pass1[k] for k in need], desc="T5 pass 2 (refine)")
        accepted = 0
        for k_pos, i_glob in enumerate(need):
            r = refined[k_pos]
            if token_edit(pass1[i_glob], r) <= REFINE_MAX_EDITS:
                pred_strs[i_glob] = r
                if r != pass1[i_glob]:
                    accepted += 1
        print(f"Pass 2 finished in {(time.time() - t1) / 60:.1f} min | accepted refinements: {accepted}/{len(need)}")
    else:
        print("No samples to refine.")

    metrics_t5, examples_t5 = evaluate_predictions(
        f"FLAN-T5 enhanced (t5max, copy_gate={manifest.get('use_copy_gate')})",
        src_strs, tgt_strs, pred_strs,
        "t5max_eval.json", "t5max_examples.csv",
        extra={
            "model_dir": str(MODEL_DIR),
            "training_manifest": manifest,
            "generation_kwargs": GEN_KWARGS,
            "iterative_refinement": True,
            "full_test_set": True,
        },
    )

## Standard benchmarks — BEA-2019 dev, JFLEG, CoNLL-2014

In-domain numbers on synthetic C4-200M noise cannot be compared to the literature. The fields' standard benchmarks are:

- **BEA-2019 (W&I+LOCNESS)** — the main shared-task benchmark. The *test* set is blind (scored only via CodaLab), so the universal public proxy is the **dev set** (4,384 sentences, gold M2 annotations). We score with the **official ERRANT span-based F0.5** (`errant_compare` against `ABCN.dev.gold.bea19.m2`) plus our full metric suite (GLEU, BLEU, behaviour categories) against references reconstructed by applying the gold edits.
- **JFLEG** — 747 test sentences with 4 human fluency-rewrite references each; the standard metric is multi-reference GLEU.
- **CoNLL-2014** (optional) — 1,312 test sentences. *Caveat for the paper:* official CoNLL numbers use the M2Scorer; we score with ERRANT, which is systematically a few points lower — label the table accordingly.

**Expectation management:** our model is trained purely on *synthetic* web-text noise. BEA-2019/CoNLL contain *human learner* errors — a real domain shift. Published systems that score 55–70 F0.5 on BEA dev all fine-tune on human-annotated corpora (W&I train, NUCLE, Lang-8) *after* synthetic pre-training; C4-200M-style data alone typically yields much lower absolute scores. For the paper, this is not a weakness but a finding: it quantifies how far synthetic-only training transfers to real learner text, and motivates the standard pretrain-then-finetune recipe as future work.

These cells need Kaggle internet access enabled (Settings → Internet → On).

In [ ]:
# BEA-2019 dev (W&I+LOCNESS): official ERRANT span-based F0.5 + full metric suite
def read_m2(path):
    """Parse an M2 file -> list of (src_sentence, {annotator_id: [(start, end, correction), ...]})."""
    blocks = Path(path).read_text(encoding="utf-8").strip().split("\n\n")
    out = []
    for block in blocks:
        lines = block.split("\n")
        src = lines[0][2:].strip()
        edits = {}
        for ln in lines[1:]:
            if not ln.startswith("A "):
                continue
            fields = ln[2:].split("|||")
            span = fields[0].split()
            s, e = int(span[0]), int(span[1])
            etype, cor = fields[1], fields[2]
            ann = int(fields[-1])
            edits.setdefault(ann, [])
            if etype != "noop":
                edits[ann].append((s, e, cor))
        out.append((src, edits))
    return out

def apply_m2_edits(src, edits):
    toks = src.split()
    offset = 0
    for s, e, cor in sorted(edits, key=lambda x: (x[0], x[1])):
        cor_toks = cor.split() if cor and cor != "-NONE-" else []
        toks[s + offset:e + offset] = cor_toks
        offset += len(cor_toks) - (e - s)
    return " ".join(toks)

def spacy_tokenize_all(texts, desc="tokenising"):
    """Re-tokenise model output with ERRANT's spaCy pipeline so hypothesis and
    gold live in the same token space."""
    annotator = _try_load_errant()
    if annotator is None:
        return [str(t) for t in texts]
    return [" ".join(tok.text for tok in annotator.parse(str(t), tokenise=True))
            for t in tqdm(texts, desc=desc)]

def errant_official_scores(orig_sents, hyp_sents_tok, gold_m2_path, work_dir, tag):
    """Write orig/hyp, build hypothesis M2 with errant_parallel, score with errant_compare."""
    orig_path = work_dir / f"{tag}.orig.txt"
    hyp_path = work_dir / f"{tag}.hyp.txt"
    hyp_m2 = work_dir / f"{tag}.hyp.m2"
    orig_path.write_text("\n".join(s.replace("\n", " ") for s in orig_sents) + "\n", encoding="utf-8")
    hyp_path.write_text("\n".join(s.replace("\n", " ") for s in hyp_sents_tok) + "\n", encoding="utf-8")
    r1 = subprocess.run(["errant_parallel", "-orig", str(orig_path), "-cor", str(hyp_path), "-out", str(hyp_m2)],
                        capture_output=True, text=True)
    if r1.returncode != 0:
        raise RuntimeError(f"errant_parallel failed:\n{r1.stderr[-2000:]}")
    r2 = subprocess.run(["errant_compare", "-hyp", str(hyp_m2), "-ref", str(gold_m2_path)],
                        capture_output=True, text=True)
    print(r2.stdout)
    nums = re.findall(r"^\s*(\d+)\s+(\d+)\s+(\d+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s*$", r2.stdout, flags=re.M)
    if not nums:
        raise RuntimeError(f"Could not parse errant_compare output:\n{r2.stdout}")
    tp, fp, fn, p, r, f = nums[-1]
    return {"tp": int(tp), "fp": int(fp), "fn": int(fn),
            "precision": float(p), "recall": float(r), "f0.5": float(f)}

_trained = MANIFEST_PATH.exists() and load_json(MANIFEST_PATH).get("completed")
if not RUN_BEA19:
    print("RUN_BEA19=False, skipping.")
elif not _trained:
    print("Training not finished yet -- skipping BEA-2019 evaluation this session.")
else:
    try:
        BEA_URL = "https://www.cl.cam.ac.uk/research/nl/bea2019st/data/wi+locness_v2.1.bea19.tar.gz"
        bea_root = OUT / "bea19"
        bea_root.mkdir(exist_ok=True)
        gold_m2 = bea_root / "wi+locness" / "m2" / "ABCN.dev.gold.bea19.m2"
        if not gold_m2.exists():
            tar_path = bea_root / "wi+locness_v2.1.bea19.tar.gz"
            print(f"Downloading {BEA_URL} ...")
            urllib.request.urlretrieve(BEA_URL, tar_path)
            with tarfile.open(tar_path) as tar:
                tar.extractall(bea_root)
        entries = read_m2(gold_m2)
        bea_srcs = [src for src, _ in entries]
        bea_refs = [apply_m2_edits(src, ed.get(0, [])) for src, ed in entries]
        print(f"BEA-2019 dev: {len(bea_srcs):,} sentences")

        bea_preds_raw = t5_correct_batch(bea_srcs, desc="T5 on BEA-2019 dev")
        bea_preds = spacy_tokenize_all(bea_preds_raw, desc="tokenising hypotheses")

        official = errant_official_scores(bea_srcs, bea_preds, gold_m2, bea_root, "bea_dev")
        print(f"Official ERRANT span-based: P={official['precision']:.4f} "
              f"R={official['recall']:.4f} F0.5={official['f0.5']:.4f}")

        metrics_bea, _ = evaluate_predictions(
            "FLAN-T5 enhanced on BEA-2019 dev (W&I+LOCNESS)",
            bea_srcs, bea_refs, bea_preds,
            "t5max_bea19_dev.json", "t5max_bea19_examples.csv",
            extra={
                "benchmark": "BEA-2019 dev (W&I+LOCNESS v2.1)",
                "official_errant_compare": official,
                "note": ("Official span-based scoring vs ABCN.dev.gold.bea19.m2. "
                         "BEA-2019 *test* is blind (CodaLab); dev is the standard public proxy. "
                         "GLEU/BLEU/behaviour computed vs annotator-0 references."),
            },
        )
    except Exception as exc:
        print(f"BEA-2019 evaluation failed (is Kaggle internet enabled?): {exc!r}")

In [ ]:
# JFLEG (multi-reference GLEU) + optional CoNLL-2014
_trained = MANIFEST_PATH.exists() and load_json(MANIFEST_PATH).get("completed")
if not RUN_JFLEG:
    print("RUN_JFLEG=False, skipping.")
elif not _trained:
    print("Training not finished yet -- skipping JFLEG this session.")
else:
    try:
        jdir = OUT / "jfleg"
        jdir.mkdir(exist_ok=True)
        base = "https://raw.githubusercontent.com/keisks/jfleg/master/test/"
        names = ["test.src", "test.ref0", "test.ref1", "test.ref2", "test.ref3"]
        for name in names:
            p = jdir / name
            if not p.exists():
                urllib.request.urlretrieve(base + name, p)
        j_src = (jdir / "test.src").read_text(encoding="utf-8").strip().split("\n")
        j_refs = [ (jdir / f"test.ref{k}").read_text(encoding="utf-8").strip().split("\n") for k in range(4) ]
        assert all(len(r) == len(j_src) for r in j_refs)
        print(f"JFLEG test: {len(j_src):,} sentences, 4 references each")

        j_preds_raw = t5_correct_batch(j_src, desc="T5 on JFLEG")
        j_preds = spacy_tokenize_all(j_preds_raw, desc="tokenising hypotheses")

        ref_lists = [[j_refs[k][i].split() for k in range(4)] for i in range(len(j_src))]
        j_gleu = corpus_gleu(ref_lists, [p.split() for p in j_preds])
        j_gleu_src = corpus_gleu(ref_lists, [s.split() for s in j_src])
        print(f"JFLEG GLEU: {j_gleu:.4f} (do-nothing source baseline: {j_gleu_src:.4f})")
        save_json({
            "model": "FLAN-T5 enhanced (t5max)",
            "notebook_version": NOTEBOOK_VERSION,
            "benchmark": "JFLEG test",
            "n_eval": len(j_src),
            "gleu_pred": float(j_gleu),
            "gleu_src_baseline": float(j_gleu_src),
        }, OUT / "t5max_jfleg.json")
        print("Saved: t5max_jfleg.json")
    except Exception as exc:
        print(f"JFLEG evaluation failed: {exc!r}")

if not RUN_CONLL14:
    print("RUN_CONLL14=False, skipping.")
elif not _trained:
    print("Training not finished yet -- skipping CoNLL-2014 this session.")
else:
    try:
        cdir = OUT / "conll14"
        cdir.mkdir(exist_ok=True)
        c_m2 = cdir / "conll14st-test-data" / "noalt" / "official-2014.combined.m2"
        if not c_m2.exists():
            url = "https://www.comp.nus.edu.sg/~nlp/conll14st/conll14st-test-data.tar.gz"
            tar_path = cdir / "conll14st-test-data.tar.gz"
            print(f"Downloading {url} ...")
            urllib.request.urlretrieve(url, tar_path)
            with tarfile.open(tar_path) as tar:
                tar.extractall(cdir)
        entries = read_m2(c_m2)
        c_srcs = [src for src, _ in entries]
        c_refs = [apply_m2_edits(src, ed.get(0, [])) for src, ed in entries]
        print(f"CoNLL-2014 test: {len(c_srcs):,} sentences")

        c_preds_raw = t5_correct_batch(c_srcs, desc="T5 on CoNLL-2014")
        c_preds = spacy_tokenize_all(c_preds_raw, desc="tokenising hypotheses")

        official = errant_official_scores(c_srcs, c_preds, c_m2, cdir, "conll14")
        print(f"ERRANT span-based: P={official['precision']:.4f} "
              f"R={official['recall']:.4f} F0.5={official['f0.5']:.4f}")
        evaluate_predictions(
            "FLAN-T5 enhanced on CoNLL-2014 test",
            c_srcs, c_refs, c_preds,
            "t5max_conll14.json", "t5max_conll14_examples.csv",
            extra={
                "benchmark": "CoNLL-2014 test",
                "official_errant_compare": official,
                "note": ("Scored with ERRANT, NOT the official M2Scorer; ERRANT runs a few "
                         "points lower -- not directly comparable to published CoNLL numbers."),
            },
        )
    except Exception as exc:
        print(f"CoNLL-2014 evaluation failed: {exc!r}")

## Final comparison vs the v1 paper model

The table and figures below put the enhanced model next to the v1 FLAN-T5-small result that is currently in the paper.

**Read with care:** v1 was evaluated on a 5,000-pair sample of the *noisy* v1 test split; v2 is evaluated on the full 25,000-pair *cleaned* test split. The two test distributions are not identical — that is the point (cleaner references raise the ceiling) — so the paper should present this as *"v2 protocol"* with the do-nothing source baseline reported next to each number, plus the BEA-2019/JFLEG rows, which **are** directly comparable to the literature.

In [ ]:
# Final comparison: v1 paper model vs enhanced v2 model
if not (OUT / "t5max_eval.json").exists():
    print("t5max_eval.json not found (training/eval not finished) -- skipping comparison this session.")
else:
    V1_RESULTS = {  # FLAN-T5-small from deepfinalproject.ipynb (t5_eval.json, tweaked_2026_05_14)
        "model": "v1: FLAN-T5-small (1.15M noisy pairs, no copy gate)",
        "n_eval": 5000,
        "gleu_src_baseline": 0.6137,
        "gleu_pred": 0.6359,
        "bleu_pred": 0.6368,
        "exact_match": 0.0564,
        "helpful_pct": 37.62,
        "harm_pct": 35.40,
        "precision_edit": 0.4198,
        "recall_edit": 0.1714,
        "f0.5_edit": 0.3255,
    }

    new = load_json(OUT / "t5max_eval.json")
    rows = []
    for label, m in (("v1 (paper)", V1_RESULTS), ("v2 (this notebook)", new)):
        rows.append({
            "run": label,
            "model": m["model"],
            "n_eval": m["n_eval"],
            "GLEU": round(m["gleu_pred"], 4),
            "GLEU src baseline": round(m["gleu_src_baseline"], 4),
            "GLEU - baseline": round(m["gleu_pred"] - m["gleu_src_baseline"], 4),
            "BLEU": round(m["bleu_pred"], 4),
            "exact %": round(m["exact_match"] * 100, 2),
            "helpful %": round(m["helpful_pct"], 2),
            "harm %": round(m["harm_pct"], 2),
            "P_edit": round(m["precision_edit"], 4),
            "R_edit": round(m["recall_edit"], 4),
            "F0.5": round(m["f0.5_edit"], 4),
        })
    comp = pd.DataFrame(rows)
    comp.to_csv(OUT / "t5max_final_comparison.csv", index=False)
    print(comp.to_string(index=False))

    bench_rows = []
    for fname, bench in [("t5max_bea19_dev.json", "BEA-2019 dev"),
                         ("t5max_jfleg.json", "JFLEG"),
                         ("t5max_conll14.json", "CoNLL-2014")]:
        p = OUT / fname
        if p.exists():
            m = load_json(p)
            off = m.get("official_errant_compare") or {}
            bench_rows.append({
                "benchmark": bench,
                "n": m.get("n_eval"),
                "GLEU": round(m["gleu_pred"], 4) if "gleu_pred" in m else None,
                "GLEU src baseline": round(m["gleu_src_baseline"], 4) if "gleu_src_baseline" in m else None,
                "official P": off.get("precision"),
                "official R": off.get("recall"),
                "official F0.5": off.get("f0.5"),
            })
    if bench_rows:
        bench_df = pd.DataFrame(bench_rows)
        bench_df.to_csv(OUT / "t5max_benchmarks.csv", index=False)
        print("\nStandard benchmarks:")
        print(bench_df.to_string(index=False))

    # Figures
    labels = ["v1 (paper)", "v2 (enhanced)"]
    colors = ["#999999", "#cc4444"]

    plt.figure(figsize=(7, 4))
    vals = [V1_RESULTS["gleu_pred"], new["gleu_pred"]]
    bases = [V1_RESULTS["gleu_src_baseline"], new["gleu_src_baseline"]]
    x = np.arange(2)
    plt.bar(x, vals, width=0.5, color=colors)
    for i, b in enumerate(bases):
        plt.hlines(b, i - 0.32, i + 0.32, linestyles="--", color="black",
                   label="do-nothing baseline" if i == 0 else None)
    for i, v in enumerate(vals):
        plt.text(i, v + 0.004, f"{v:.3f}", ha="center")
    plt.xticks(x, labels); plt.ylabel("GLEU"); plt.title("GLEU vs do-nothing baseline (own test split)")
    plt.legend(); plt.grid(alpha=0.3, axis="y"); plt.tight_layout()
    plt.savefig(OUT / "t5max_08_gleu_comparison.png", dpi=120, bbox_inches="tight"); plt.show()

    plt.figure(figsize=(7, 4))
    w = 0.35
    plt.bar(x - w / 2, [V1_RESULTS["helpful_pct"], new["helpful_pct"]], w, label="helpful %", color="#4477cc")
    plt.bar(x + w / 2, [V1_RESULTS["harm_pct"], new["harm_pct"]], w, label="harm %", color="#cc8844")
    plt.xticks(x, labels); plt.ylabel("%"); plt.title("Helpful vs harm")
    plt.legend(); plt.grid(alpha=0.3, axis="y"); plt.tight_layout()
    plt.savefig(OUT / "t5max_09_helpful_harm.png", dpi=120, bbox_inches="tight"); plt.show()

    plt.figure(figsize=(7, 4))
    vals = [V1_RESULTS["f0.5_edit"], new["f0.5_edit"]]
    plt.bar(x, vals, width=0.5, color=colors)
    for i, v in enumerate(vals):
        plt.text(i, v + 0.005, f"{v:.3f}", ha="center")
    plt.xticks(x, labels); plt.ylabel("edit-level F0.5 (ERRANT)"); plt.title("Edit-level F0.5")
    plt.grid(alpha=0.3, axis="y"); plt.tight_layout()
    plt.savefig(OUT / "t5max_10_f05_comparison.png", dpi=120, bbox_inches="tight"); plt.show()

    print("Saved: t5max_final_comparison.csv, t5max_benchmarks.csv, t5max_08/09/10 PNGs")

In [ ]:
# Export all result artifacts into one ZIP (excludes model weights and checkpoints)
ZIP_PATH = OUT / "t5max_results_export.zip"
patterns = ["t5max_*.json", "t5max_*.csv", "t5max_*.png", "c4_gec_clean_meta.json"]
files_to_zip = sorted({p for pat in patterns for p in OUT.glob(pat)})
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in files_to_zip:
        zf.write(p, p.name)
        print(f"  + {p.name}")
print(f"\nSaved {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:.1f} MB)")
print("\nNext step for the paper: download this ZIP and share t5max_eval.json, "
      "t5max_bea19_dev.json, t5max_jfleg.json, t5max_final_comparison.csv and the PNGs.")